In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize
from scipy.special import expit as sigmoid
from scipy.special import logit
from pathlib import Path
import seaborn as sns
from numba import jit, njit
from joblib import Parallel, delayed
from scipy.optimize import minimize, fsolve
from functools import partial
import cma
from sklearn.preprocessing import PolynomialFeatures

COMP_NAME = "g2net-gravitational-wave-detection"
# INPUT_PATH = f"../input/{COMP_NAME}/"
# OUTPUT_PATH = f"../input/ensemble-0916/"
INPUT_PATH = "/mnt/storage_dimm2/kaggle_data/g2net-gravitational-wave-detection"
OUTPUT_PATH = "/mnt/storage_dimm2/kaggle_data/g2net-gravitational-wave-detection/stacking"

In [ ]:
gt = pd.read_csv(INPUT_PATH + "/" + "training_labels.csv")
names = [
    "A-RN200",
    "A-B5PL",
    "A-B5",
    "A-V2M",
    "A-1D",
    "A-CWT",
    "A-B3",
    "A-B3CWT",
    "I-B3",
    "I-B4",
    "I-B5",
    "I-B7",
    "I-IV3",
    "R-1D",
    "R-V2SD",
    "R-V2SD2",
    "R-V2",
    "R-V2SDCBAM",
    "R-133",  # "R-137",
    "V-V2SD",
    "V_3D",
    "M-1D",
    "M-1DS32",
    "M-1DC16",
    "M-SD16",
    "M-SD32",
]

model_names = [
    "resnet-200d",
    "eff-B5",
    "eff-b5",
    "ef-v2M",
    "1dCNN",
    "CWT",
    "B3",
    "B3CWT",
]
oof_files = [
    "oofs_20210830-191530_CV-0.87733.csv",  # ResNet200d
    "oofs_20210918-112751_CV-0.87870.csv",  # EfficientNet B5
    "oofs_20210901-130913_CV-0.87749.csv",  # EfficientNet B5
    "oofs_20210902-200442_CV-0.87733.csv",  # EfficientNet V2 M
    "oofs_20210915-003751_CV-0.87393.csv",  # 1D CNN
    "oofs_20210915-072731_CV-0.87707.csv",  # CWT
    "oofs_20210923-103832_CV-0.87829.csv",  # B3
    "oofs_20210925-085935_CV-0.87818.csv",  # B3-CWT
]

oof_paths = [
    OUTPUT_PATH
    + "/"
    + (model_names[i] + "-" + f.split("_")[1] + "_predictions")
    + "/"
    + f
    for i, f in enumerate(oof_files)
]
sub_paths = [
    OUTPUT_PATH
    + "/"
    + (model_names[i] + "-" + f.split("_")[1] + "_predictions")
    + "/"
    + f.replace("oofs", "sub")
    for i, f in enumerate(oof_files)
]
# test_paths = [OUTPUT_PATH / (model_names[i] + "-" + f.split("_")[1]+"_predictions") / f.replace("oofs", "test") for i,f in enumerate(oof_files)]

oof_paths_other = [
    OUTPUT_PATH + "/oof_effB3-512-PL-float.csv",
    OUTPUT_PATH + "/oof_effB4-512-PL-CV087755.csv",
    OUTPUT_PATH + "/oof_effB5-512-PL-CV087728.csv",
    OUTPUT_PATH + "/oof_w_effB7-512-SKF.csv",
    OUTPUT_PATH + "/oof_inception_v3_PL_CV087466.csv",
    OUTPUT_PATH + "/oof_1DCNN_richard.csv",
    OUTPUT_PATH + "/oof_112nd_V2SD_PL.csv",
    OUTPUT_PATH + "/oof_121st_V2SD.csv",
    OUTPUT_PATH + "/oof_122nd_V2.csv",
    OUTPUT_PATH + "/oof_124th_V2SDCBAM_tta_blend.csv",
    OUTPUT_PATH + "/oof_133rd_V2SD_PL_4ep_2em3lr_32ch_vf_sc01_drop05.csv",
    # OUTPUT_PATH + "/oof_137th_tta_blend.csv",
    # OUTPUT_PATH +  "/oof_V2_c16_sGW_vflip_sc01_PL_CV88163.csv",
    OUTPUT_PATH + "/oof_V2SD_sGW_vflip_sc01_PL_CV88003.csv",
    # OUTPUT_PATH +  "/oof_resnet34-sGW2ep-PL-sc01_CV87764.csv",
    OUTPUT_PATH + "/oof_1D2D_1ep_CV88153.csv",
    OUTPUT_PATH + "/oof_1D_adjust_MC_CV88158.csv",
    OUTPUT_PATH + "/oof_1D_adjust_Split32_MC_CV88149.csv",
    OUTPUT_PATH + "/oof_1D_adjust_c16_MC_CV88170.csv",
    OUTPUT_PATH + "/oof_1D_adjust_SD16_MC_CV88164.csv",
    OUTPUT_PATH + "/oof_1D_adjust_SD32_MC_CV88185.csv",
]

sub_paths_other = [
    OUTPUT_PATH + "/submission_effB3-512-PL-CV0877245.csv",
    OUTPUT_PATH + "/submission_effB4-512-PL-CV087755.csv",
    OUTPUT_PATH + "/submission_effB5-512-PL-CV087728.csv",
    OUTPUT_PATH + "/submission_w_effB7-512-SKF.csv",
    OUTPUT_PATH + "/submission_inception_v3_PL_CV087466.csv",
    OUTPUT_PATH + "/submission_main_35th_GeM_vflip_shuffle01.csv",
    OUTPUT_PATH + "/submission_112nd_V2SD_PL.csv",
    OUTPUT_PATH + "/submission_121st_V2SD.csv",
    OUTPUT_PATH + "/submission_122nd_V2.csv",
    OUTPUT_PATH + "/submission_124th_V2SDCBAM_tta_blend.csv",
    OUTPUT_PATH + "/submission_133rd_V2SD_PL_4ep_2em3lr_32ch_vf_sc01_drop05.csv",
    # OUTPUT_PATH + "/submission_137th_tta_blend.csv",
    # OUTPUT_PATH +  "/submission_V2_c16_sGW_vflip_sc01_PL_CV88163.csv",
    OUTPUT_PATH + "/submission_V2SD_sGW_vflip_sc01_PL_CV88003.csv",
    # OUTPUT_PATH +  "/submission_resnet34-sGW2ep-PL-sc01_CV87764.csv",
    OUTPUT_PATH + "/submission_1D2D_1ep_CV88153.csv",
    OUTPUT_PATH + "/submission_1D_adjust_MC_CV88158.csv",
    OUTPUT_PATH + "/submission_1D_adjust_Split32_MC_CV88149.csv",
    OUTPUT_PATH + "/submission_1D_adjust_c16_MC_CV88170.csv",
    OUTPUT_PATH + "/submission_1D_adjust_SD16_MC_CV88164.csv",
    OUTPUT_PATH + "/submission_1D_adjust_SD32_MC_CV88185.csv",
]

oof_paths.extend(oof_paths_other)
sub_paths.extend(sub_paths_other)

oof_list = [pd.read_csv(p) for p in oof_paths]
sub_list = [pd.read_csv(p) for p in sub_paths]
# test_list = [pd.read_csv(p) for p in test_paths]

oof1 = oof_list[0]
sub1 = sub_list[0]
# test1 = test_list[0]

oof_df = oof1[["id", "target", "fold"]].copy()
for i, oof in enumerate(oof_list):
    if "effB" in oof_paths[i] or "inception_v3" in oof_paths[i]:
        # oof from Isamu
        oof_df = oof_df.merge(
            oof[["id", "y_pred"]].rename(columns={"y_pred": names[i]}),
            on=["id"],
            how="left",
        )
    else:
        oof_df = oof_df.merge(
            oof[["id", "prediction"]].rename(columns={"prediction": names[i]}),
            on=["id"],
            how="left",
        )

sub_df = sub1[["id"]].copy()
for i, sub in enumerate(sub_list):
    sub_df = sub_df.merge(
        sub[["id", "target"]].rename(columns={"target": names[i]}),
        on=["id"],
        how="left",
    )
print("Finish Loading Data")

In [ ]:
oof_df.head()

In [ ]:
oof_df.describe()

In [ ]:
# pre_method = "rank"
# pre_method = "logit"
# pre_method = "prob**2"
pre_method = "prob"

clip = 20
eps = 1e-9
for i, col in enumerate(oof_df.columns[3:]):
    if pre_method == "rank":
        oof_df[col] = oof_df[col].rank(pct=True)
        sub_df[col] = sub_df[col].rank(pct=True)
    elif pre_method == "rank-power2":
        oof_df[col] = oof_df[col].rank(pct=True) ** 2
        sub_df[col] = sub_df[col].rank(pct=True) ** 2
    elif pre_method == "rank-fold":
        oof_df[col] = oof_df.groupby("fold")[col].rank(pct=True)
        sub_df[col] = sub_df[col].rank(pct=True)
    elif pre_method == "logit":
        if oof_df[col].max() > 1:
            oof_df[col] = np.clip(oof_df[col], -clip, clip)
            sub_df[col] = np.clip(sub_df[col], -clip, clip)
            pass
        else:
            oof_df[col] = np.log(oof_df[col]) - np.log(1 - oof_df[col] + eps)
            sub_df[col] = np.log(sub_df[col]) - np.log(1 - sub_df[col] + eps)
    elif pre_method == "prob":
        if oof_df[col].max() <= 1 and oof_df[col].min() >= 0:
            continue
        else:
            oof_df[col] = np.exp(oof_df[col]) / (1 + np.exp(oof_df[col]))
            sub_df[col] = np.exp(sub_df[col]) / (1 + np.exp(sub_df[col]))
    elif pre_method == "prob**2":
        if oof_df[col].max() <= 1 and oof_df[col].min() >= 0:
            oof_df[col] = oof_df[col] ** 2
            sub_df[col] = sub_df[col] ** 2
        else:
            oof_df[col] = (np.exp(oof_df[col]) / (1 + np.exp(oof_df[col]))) ** 2
            sub_df[col] = (np.exp(sub_df[col]) / (1 + np.exp(sub_df[col]))) ** 2

In [ ]:
oof_df.describe()

In [ ]:
@jit(nopython=True)
def fast_auc(y_true, y_prob):
    y_true = np.asarray(y_true)
    y_true = y_true[np.argsort(y_prob)]
    nfalse = 0
    auc = 0
    n = len(y_true)
    for i in range(n):
        y_i = y_true[i]
        nfalse += 1 - y_i
        auc += y_i * nfalse
    auc /= nfalse * (n - nfalse)
    return auc


@njit
def _auc(actual, pred_ranks):
    actual = np.asarray(actual)
    pred_ranks = np.asarray(pred_ranks)
    n_pos = np.sum(actual)
    n_neg = len(actual) - n_pos
    return (np.sum(pred_ranks[actual == 1]) - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


def auc(actual, predicted):
    pred_ranks = rankdata(predicted)
    return _auc(actual, pred_ranks)


def score_oof(oof, full=False, col=None, get_list=False):
    if col is None:
        col = "prediction"
    if full:
        return round(roc_auc_score(oof["target"], oof[col]), 6)
    else:
        scores = [roc_auc_score(df["target"], df[col]) for _, df in oof.groupby("fold")]
        if get_list:
            return scores
        return round(np.mean(scores), 6)


def score_oof_fast(oof, full=False, col=None, get_list=False):
    if col is None:
        col = "prediction"
    if full:
        return round(fast_auc(oof["target"].values, oof[col].values), 6)
    else:
        scores = [fast_auc(df["target"].values, df[col].values) for _, df in oof.groupby("fold")]
        if get_list:
            return scores
        return round(np.mean(scores), 6)

In [ ]:
# drop_list = ["I-B3", "I-B4", "A-V2M", "A-B5"]

In [ ]:
cma_opt_preds = oof_df.copy()
model_cols = oof_df.columns[3:]
# model_cols = [x for x in model_cols if x not in drop_list]
X = cma_opt_preds[model_cols].values

poly = PolynomialFeatures(2, interaction_only=False, include_bias=False)
X = poly.fit_transform(X)

print(model_cols, cma_opt_preds.shape, X.shape)

def get_score(weights, oof=cma_opt_preds):
    weights = np.array(weights)
    cma_opt_preds["prediction"] = (weights * X).sum(axis=1)
    return -score_oof_fast(cma_opt_preds)

init_guess = [1 / X.shape[1]] * X.shape[1]
print("Inital Blend OOF: {:.6f}".format(-get_score(init_guess)))

es = cma.CMAEvolutionStrategy(init_guess, 0.5, {'seed': 2021, "bounds": [None, None], "tolfun": 1e-6, "timeout": "3.0 * 60**2"})
es.optimize(get_score)
print(es.result_pretty())
print("Optimised Blend OOF: {:.6f}".format(-get_score(es.result.xbest)))

ws = es.result.xbest

cma_cv_score = -get_score(es.result.xbest)
cma.plot();

In [ ]:
ws = es.result.xbest

cma_cv_score = -get_score(es.result.xbest)
cma_cv_score

In [ ]:
pd.DataFrame(dict(zip(model_cols, ws)), index=["weight"]).T.plot(
    kind="barh", figsize=(10, 16)
);

In [ ]:
cma_opt_preds["prediction"] = np.sum(ws * X, axis=1)
cma_opt_preds[["id", "target", "fold", "prediction"]].to_csv(
    f"./oof_pred_cma-es_CV{cma_cv_score:.5f}.csv", index=False
)

sub_cma = sub_df[["id"]].copy()
sub_cma["target"] = np.sum(poly.fit_transform(sub_df.iloc[:,1:]) * ws, axis=1)
sub_cma.to_csv(f"./cma-es_ensemble_{cma_cv_score:0.5f}.csv", index=False)
sub_cma["target"].hist(bins=100, figsize=(10, 4))

In [ ]:
# 0.882952 Logits
# 0.882911 Rank

# AUC Pruning

In [ ]:
class OptimizeAUC:
    """
    Class for optimizing AUC.
    This class is all you need to find best weights for
    any model and for any metric and for any types of predictions.
    With very small changes, this class can be used for optimization of
    weights in ensemble models of _any_ type of predictions
    """

    def __init__(self):
        self.coef_ = 0

    def _auc(self, coef, X, y):
        """
        This functions calulates and returns AUC.
        :param coef: coef list, of the same length as number of models
        :param X: predictions, in this case a 2d array
        :param y: targets, in our case binary 1d array
        """
        x_coef = X * coef
        predictions = np.sum(x_coef, axis=1)

        #         auc_score = roc_auc_score(y, predictions)
        auc_score = fast_auc(y.values, predictions.values)
        return -1.0 * auc_score

    def fit(self, X, y):
        loss_partial = partial(self._auc, X=X, y=y)

        # dirichlet distribution. you can use any distribution you want
        # to initialize the coefficients
        # we want the coefficients to sum to 1
        # initial_coef = np.random.dirichlet(np.ones(X.shape[1]), size=1)
        initial_coef = np.ones(X.shape[1]) / X.shape[1]

        es = cma.CMAEvolutionStrategy(
            initial_coef, 0.5, {"seed": 2021, "bounds": [None, None], "maxfevals": 2500, "tolfun": 1e-6}
        )
        es.optimize(loss_partial)
        self.coef_ = es.result.xbest

    def predict(self, X):
        x_coef = X * self.coef_
        predictions = np.sum(x_coef, axis=1)
        return predictions

In [ ]:
def scorer_cma(oofs, col_names, parallel=False):
    
    def score_fold(fold):
        trn_df = oofs.query(f"fold != {fold}")
        val_df = oofs.query(f"fold == {fold}")
        
        bnds = [(-1, 1) for _ in range(len(col_names))]

        opt = OptimizeAUC()
        opt.fit(trn_df[col_names], trn_df["target"])
        y_pred = opt.predict(val_df[col_names])
        return roc_auc_score(val_df["target"], y_pred)
        
    if parallel:
        fold_scores = Parallel(n_jobs=5)(delayed(score_fold)(fold) for fold in range(5))
    else:
        fold_scores = [score_fold(fold) for fold in range(5)]

    return np.mean(fold_scores)

In [ ]:
# scorer_cma(oof_df, oof_df.columns[3:], True)

In [ ]:
def pruning(oofs, candidates):
    history = []
    score = scorer_cma(oofs, candidates, True)
    print(f"Initial score {score:0.6f}")

    while len(candidates) > 2:
        trial_candidates = [
            candidates[:i] + candidates[i + 1 :] for i, _ in enumerate(candidates)
        ]
        scores = Parallel(n_jobs=32)(delayed(scorer_cma)(oofs, tc) for tc in trial_candidates)
        removed = candidates[np.argmax(scores)]
        del candidates[np.argmax(scores)]
        score = scorer_cma(oofs, candidates, True)
        history.append(
            {
                "models": candidates.copy(),
                "score": score,
                "removed": removed,
            }
        )
        print(
            f"{len(history)} New score {score:0.6f}",
            f"Removed {removed}. {len(candidates)} models",
        )

    history = pd.DataFrame(history)
    history.to_csv("pruning_cma-es.csv", index=False)
    print(history.tail(40))

In [ ]:
# pruning(oof_df, list(oof_df.columns[3:]))

```
Initial score 0.882921
1 New score 0.882923 Removed I-B3. 22 models
2 New score 0.882926 Removed A-V2M. 21 models
3 New score 0.882929 Removed A-B5. 20 models
4 New score 0.882931 Removed I-B4. 19 models
5 New score 0.882931 Removed I-IV3. 18 models
6 New score 0.882928 Removed A-RN200. 17 models
7 New score 0.882926 Removed V_3D. 16 models
8 New score 0.882925 Removed R-1D. 15 models
9 New score 0.882921 Removed A-CWT. 14 models
10 New score 0.882916 Removed I-B5. 13 models
11 New score 0.882912 Removed M-1D. 12 models
12 New score 0.882906 Removed A-B3CWT. 11 models
13 New score 0.882898 Removed V-V2SD. 10 models
14 New score 0.882887 Removed A-1D. 9 models
15 New score 0.882875 Removed A-B3. 8 models
16 New score 0.882866 Removed R-V2SD2. 7 models
```